In [ ]:
import os
import pickle
from pennylane import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import pennylane as qml
import numpy as np

import pandas as pd
import json
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import time
import copy
from collections import Counter


## Configuration

In [ ]:
DATA_PATH = "dataset/mirage2019_LOPEZ_lopez_lopez_36P_4F_APP_xST_PAD_metadata.pickle"
BATCH_SIZE = 50
EPOCHS = 100
LEARNING_RATE = 1e-3
ENABLE_EARLY_STOPPING = False
PATIENCE = 15 # Early stopping patience
RANDOM_SEED = 2025

N_PACKETS = 10
N_FEATURES = 4

N_QUBITS = 5
N_LAYERS = 3

# Riproducibilità
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED) # Utile per hash di dizionari/set

# 2. PyTorch (CPU e GPU)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

# 3. Determinismo su GPU (CuDNN)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Load Data

In [ ]:
print("Loading dataset...")
with open(DATA_PATH, "rb") as f:
    X_raw = np.array(pickle.load(f), dtype=np.float32) # Convert to numpy array
    y_raw = np.array(pickle.load(f)) # Convert to numpy array

print(f"Data shape: {X_raw.shape}")
print(f"Labels shape: {len(y_raw)}")

# Example sample
example_sample = 5000
print("\nExample Sample (First 5 packets):\n", X_raw[example_sample][:5])
print("Example Label:", y_raw[example_sample])

## Label Encoding

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)
NUM_CLASSES = len(le.classes_)
print(f"Number of classes: {NUM_CLASSES}")
print("Classes:", le.classes_)

## Train-Validation-Test split

In [ ]:
TRAIN_SIZE = 0.7
VAL_SIZE = 0.15
TEST_SIZE = 0.15

# Check if sizes sum to 1
assert TRAIN_SIZE + VAL_SIZE + TEST_SIZE == 1.0, "Train, Val, Test sizes must sum to 1."

# First split: 85% Train+Val, 15% Test
X_temp, X_test, y_temp, y_test = train_test_split(X_raw, y_encoded, test_size=TEST_SIZE, stratify=y_encoded, random_state=2025)

# Second split: Split the 85% into Train (70% total) and Val (15% total)
# 15 / 85 ~= 0.1765
tmp_size = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=tmp_size, stratify=y_temp, random_state=2025)

# # For testing resize training to 1000 samples
# X_train = X_train[:1000]
# y_train = y_train[:1000]
# X_val = X_val[:500]
# y_val = y_val[:500]
# X_test = X_test[:500]
# y_test = y_test[:500]

print(f"Train shape: {X_train.shape} ({len(X_train)/len(X_raw):.1%})")
print(f"Val shape:   {X_val.shape}   ({len(X_val)/len(X_raw):.1%})")
print(f"Test shape:  {X_test.shape}  ({len(X_test)/len(X_raw):.1%})")

(Optional) Reduce Training set size

In [ ]:
# New training size
NEW_TRAIN_SIZE = 10000
if len(X_train) > NEW_TRAIN_SIZE:
    percentage = NEW_TRAIN_SIZE / len(X_train)
    # train test spit to reduce training set size
    X_train, _, y_train, _ = train_test_split(X_train, y_train, train_size=percentage, stratify=y_train, random_state=2025)
    print(f"Reduced Train shape: {X_train.shape} ({len(X_train)/len(X_raw):.1%})")
else:
    print("Training set size is less than or equal to the new size. No reduction applied.")

## Preprocessing Strategy 1

This domain-aware approach explicitly masks padding values to ensure they do not corrupt the model's learning process. It utilizes Log1p normalization to compress the dynamic range of heavy-tailed network features effectively mitigating the impact of outliers while preserving the semantic distinction between padding and actual data.

In [ ]:
def preprocess_data(X):
    """
    Handles padding and normalizes features.
    X shape: (N, 36, 4)
    Features: [DIR, PL, TCPWIN, IAT]
    """
    X_proc = X.copy().astype(np.float32)
    
    # Mask for padding (using PL at index 1)
    is_padding = (X_proc[:, :, 1] == -1)
    
    # Set all features to 0 where is_padding is True
    mask = np.repeat(is_padding[:, :, np.newaxis], 4, axis=2)
    X_proc[mask] = 0
    
    # Apply Log1p to PL (1), TCPWIN (2), IAT (3)
    X_proc[:, :, 1:] = np.log1p(np.maximum(X_proc[:, :, 1:], 0)) 
    
    return X_proc[:, :N_PACKETS, :]

print("Preprocessing data...")
X_train_proc = preprocess_data(X_train)
X_val_proc = preprocess_data(X_val)
X_test_proc = preprocess_data(X_test)

print("Preprocessing complete.")
print("Train shape:", X_train_proc.shape)
print("Val shape:", X_val_proc.shape)
print("Test shape:", X_test_proc.shape)

## Preprocessing Strategy 2

This generic method employs a standard Min-Max Scaler to linearly map all feature values into a fixed [0, 1] range without special handling for padding. Consequently, it is highly sensitive to outliers, which can cause valid traffic data to be compressed into an indistinguishably small interval, while also incorrectly treating padding markers as valid minimum values.

In [ ]:
def preprocess_data(X):
    X_proc = X.copy().astype(np.float32)
    # min max scaler
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(np.reshape(X_proc, [-1, N_FEATURES]))
    X_proc = scaler.transform(np.reshape(X_proc, [-1, N_FEATURES]))
    X_proc = np.reshape(X_proc, [-1, X.shape[1], N_FEATURES])
    return X_proc[:, :N_PACKETS, :]

print("Preprocessing data...")
X_train_proc = preprocess_data(X_train)
X_val_proc = preprocess_data(X_val)
X_test_proc = preprocess_data(X_test)

print("Preprocessing complete.")
print("Train shape:", X_train_proc.shape)
print("Val shape:", X_val_proc.shape)
print("Test shape:", X_test_proc.shape)

## Preprocessing Strategy 3

This approach prioritizes feature engineering by merging direction and size into a single "signed throughput" feature, effectively reducing dimensionality while capturing flow dynamics. However, by subsequently applying a standard Min-Max Scaler without explicit padding masking, it obscures the semantic zero-point of the signed data and remains vulnerable to outlier compression.

In [ ]:
# Combined DIR and PL into signed PL
def combine_dir_pl(input_array):
    # Extract columns based on their role
    control_column = input_array[:, :, 0] # DIR
    data_values = input_array[:, :, 1] # PL

    # Determine the multiplier: -1 if control_column is 1, 1 otherwise
    sign_multiplier = np.where(control_column == 1, -1, 1)

    # Apply the sign correction to the data values
    signed_data_values = data_values * sign_multiplier

    # Extract the remaining features
    other_features = input_array[:, :, 2:4]

    # Reshape the processed column to maintain 3D structure for concatenation
    signed_data_values_reshaped = signed_data_values[:, :, np.newaxis]

    # Concatenate the processed column with the other features
    processed_array = np.concatenate((signed_data_values_reshaped, other_features), axis=2)
    return processed_array

In [ ]:

def preprocess_data(X):
    """
    Combines DIR and PL into signed PL, handles padding, and normalizes features.
    X shape: (N, 36, 4)
    Features: [DIR, PL, TCPWIN, IAT]
    """
    X_proc = X.copy().astype(np.float32)
    # Combine DIR and PL into signed PL
    X_proc = combine_dir_pl(X_proc)
    # min max scaler
    scaler = MinMaxScaler(feature_range=(0, 1))
    # Now features are: [signed_PL, TCPWIN, IAT]
    N_FEATURES = 3
    scaler.fit(np.reshape(X_proc, [-1, N_FEATURES]))
    X_proc = scaler.transform(np.reshape(X_proc, [-1, N_FEATURES]))
    X_proc = np.reshape(X_proc, [-1, X.shape[1], N_FEATURES])
    return X_proc[:, :N_PACKETS, :]


print("Preprocessing data...")
X_train_proc = preprocess_data(X_train)
X_val_proc = preprocess_data(X_val)
X_test_proc = preprocess_data(X_test)

print("Preprocessing complete.")
print("Train shape:", X_train_proc.shape)
print("Val shape:", X_val_proc.shape)
print("Test shape:", X_test_proc.shape)

# Adjust N_FEATURES accordingly
N_FEATURES = 3
print("Adjusted N_FEATURES:", N_FEATURES)


## Preprocessing Strategy 4

combines the domain-aware padding handling and Log1p normalization of Strategy 1 with the feature engineering of Strategy 3. By merging direction and size into a signed throughput feature while explicitly masking padding values, it effectively reduces dimensionality and preserves semantic integrity, enhancing the model's ability to learn meaningful patterns from the data.

In [ ]:
def preprocess_data(X):
    """
    Handles padding and normalizes features.
    X shape: (N, 36, 4)
    Features: [DIR, PL, TCPWIN, IAT]
    """
    X_proc = X.copy().astype(np.float32)
    # Mask for padding (using PL at index 1)
    is_padding = (X_proc[:, :, 1] == -1)

    # Set all features to 0 where is_padding is True
    mask = np.repeat(is_padding[:, :, np.newaxis], 4, axis=2)
    X_proc[mask] = 0

    # Apply Log1p to PL (1), TCPWIN (2), IAT (3)
    X_proc[:, :, 1:] = np.log1p(np.maximum(X_proc[:, :, 1:], 0))
    X_proc = combine_dir_pl(X_proc)
    return X_proc[:, :N_PACKETS, :]

print("Preprocessing data...")
X_train_proc = preprocess_data(X_train)
X_val_proc = preprocess_data(X_val)
X_test_proc = preprocess_data(X_test)

print("Preprocessing complete.")
print("Train shape:", X_train_proc.shape)
print("Val shape:", X_val_proc.shape)
print("Test shape:", X_test_proc.shape)

# Adjust N_FEATURES accordingly
N_FEATURES = 3
print("Adjusted N_FEATURES:", N_FEATURES)

## Dataset & Dataloader
Pytorch Dataset wrapper.

In [ ]:
class MirageDataset(Dataset):
    def __init__(self, X, y):
        # X shape: (N, 36, 4)
        # CNN 1D expects (N, Channels, Length) -> (N, 4, 36)
        self.X = torch.FloatTensor(X).permute(0, 2, 1).double().to(DEVICE) # Remove to(DEVICE) if dataset doesn't fit in memory
        self.y = torch.LongTensor(y).to(DEVICE) # Remove to(DEVICE) if dataset doesn't fit in memory

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = MirageDataset(X_train_proc, y_train)
val_dataset = MirageDataset(X_val_proc, y_val)
test_dataset = MirageDataset(X_test_proc, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Model Selection

### Amplitude Embedding Model
A hybrid neural network that leverages Amplitude Embedding to encode classical data into the amplitudes of the quantum state. This approach maximizes data density, allowing the encoding of $2^N$ features into $N$ qubits, preceded by a sigmoid-activated classical dense layer.

In [ ]:
from nn_models import AmpHybridModel as HybridModel

### Angle Embedding Model
A streamlined hybrid model utilizing Angle Embedding, where input features are mapped directly to qubit rotation angles in a 1:1 ratio. It features a classical pre-processing layer followed by a quantum circuit with strongly entangling layers, offering a shallow and noise-resilient embedding strategy.

In [ ]:
from nn_models import AngleHybridModel as HybridModel

### Ring Model
A hybrid architecture that employs a custom Ring Embedding strategy, splitting the input into two sets of features encoded via rotations and circular CNOT entangling patterns. This design doubles the data capacity compared to simple angle embedding and introduces correlations between qubits early in the circuit.

In [ ]:
from nn_models import RingHybridModel as HybridModel

### Waterfall Model
A complex hybrid model featuring a Waterfall Embedding scheme that splits inputs into Y-rotation and Z-rotation blocks. It incorporates a dense, all-to-all "waterfall" connectivity of CNOT gates in the first stage, creating a highly entangled state before the variational ansatz layers.

In [ ]:
from nn_models import WaterfallHybridModel as HybridModel

Model instantiation

In [ ]:
model = HybridModel(
    n_qubits=N_QUBITS,
    n_layers=N_LAYERS,
    n_features=N_FEATURES,
    n_packets=N_PACKETS,
    num_classes=NUM_CLASSES
)
# Sposta il modello sul device e convertilo in double (float64) per precisione quantistica
model = model.to(DEVICE).double()
print(model.get_model_name())
print(model)

## Training Setup

In [ ]:
# Configurazione Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        # inputs, labels = inputs.to(DEVICE), labels.to(DEVICE) # Remove comment if dataset does't fit into memory

        optimizer.zero_grad()
        outputs = model(inputs)
        # loss = criterion(torch.log(outputs + 1e-10), labels)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        # scheduler.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100. * correct / total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            # inputs, labels = inputs.to(DEVICE), labels.to(DEVICE) # Remove comment if dataset does't fit into memory
            outputs = model(inputs)
            # loss = criterion(torch.log(outputs + 1e-10), labels)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100. * correct / total

### CrossEntropy Loss function

The cross-entropy loss (or log loss) is a fundamental metric in classification models, measuring the difference between the predicted probability distribution and the true distribution of the labels. It heavily penalizes predictions that are confidently wrong, guiding the model to improve its predictions by minimizing this difference during training.

In [ ]:
criterion = nn.CrossEntropyLoss()

### Wieghted CrossEntropy Loss function

Weighted CrossEntropy assigns different weights to each class based on their frequency in the training set. Classes with fewer samples receive higher weights, helping the model to pay more attention to them during training.

In [ ]:
def compute_class_weights():
    class_counts = Counter(y_train)
    total_samples = sum(class_counts.values())
    weights = []
    for i in range(NUM_CLASSES):
        count = class_counts.get(i, 0)
        if count > 0:
            weights.append(total_samples / (NUM_CLASSES * count))
        else:
            weights.append(1.0)
    return torch.FloatTensor(weights)

In [ ]:
class_weights = compute_class_weights().to(DEVICE).double()
criterion = nn.CrossEntropyLoss(weight=class_weights)

### Focal Loss function

Focal loss is designed to address class imbalance by down-weighting easy examples and focusing more on hard, misclassified examples. Weights are calculated in the same way as Weighted CrossEntropy. Gamma is a tunable focusing parameter that adjusts the rate at which easy examples are down-weighted.

In [ ]:
GAMMA = 2.0 # Focusing parameter. 2 is a common choice.
ALPHA = compute_class_weights().to(DEVICE).double()

Another strategy. Gamma is set to 3.0 to increase the focus on hard examples and ALPHA is set to uniform weights.

In [ ]:
GAMMA = 3.0
ALPHA = torch.ones(NUM_CLASSES, dtype=torch.double).to(DEVICE)

In [ ]:

# Loading Focal loss from torch hub
focal_loss = torch.hub.load(
	'adeelh/pytorch-multi-class-focal-loss',
	model='focal_loss',
	alpha=ALPHA,
	gamma=GAMMA,
	reduction='mean',
	device=DEVICE,
	dtype=torch.double,
	force_reload=False
)

criterion = focal_loss

## Training Loop

In [ ]:
# Dizionario per salvare la history
history = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}

In [ ]:
print(f"[INFO] Addestramento iniziato. {model.get_model_name()}")
start_time = time.time()

best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
patience_counter = 0

for epoch in range(EPOCHS):
    start_epoch_time = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    end_epoch_time = time.time()
    epoch_duration = end_epoch_time - start_epoch_time

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Loss: {train_loss:.4f} - Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}"
          f" | Time: {epoch_duration:.2f}s")


    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"  -> Validation loss improved. Model saved.")
    else:
        patience_counter += 1
        print(f"  -> No improvement. Patience: {patience_counter}", f"/ {PATIENCE}" if ENABLE_EARLY_STOPPING else "")

    if patience_counter >= PATIENCE and ENABLE_EARLY_STOPPING:
        print("Early stopping triggered.")
        break

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time/60:.2f} minutes.")

# Load best model weights
last_model = copy.deepcopy(model)
model.load_state_dict(best_model_wts)

Saving Results

In [ ]:
# Saving results
model_name = "10E_50B_68k_3F_log1p"
output_dir = "./cross_AMP_quantum_models"

os.makedirs(f"{output_dir}", exist_ok=True)
os.makedirs(f"{output_dir}/{model_name}", exist_ok=True)

# with open(f"{output_dir}/{model_name}/model_summary.txt", "w") as f:
#     model.summary(print_fn=lambda x: f.write(x + "\n"))

df_history = pd.DataFrame(history)
df_history.to_csv(f"{output_dir}/{model_name}/training_history.csv", index=False)

torch.save(model.state_dict(), f"{output_dir}/{model_name}/model.pth")

## Testing and Evaluation

(Optnional) Load Model

In [ ]:
output_dir = "./cross_AMP_quantum_models"
model_name = "100E_50B_30k"

model = HybridModel(
    n_qubits=N_QUBITS,
    n_layers=N_LAYERS,
    n_features=N_FEATURES,
    n_packets=N_PACKETS,
    num_classes=NUM_CLASSES
)
model = model.double()
weights = torch.load(f"{output_dir}/{model_name}/model.pth", map_location=DEVICE)
load_result = model.load_state_dict(weights)
print(f"Esito caricamento: {load_result}")
model = model.to(DEVICE)
model.eval()
print('Model loaded correctly')


# Load training history from csv
df_history = pd.read_csv(f"{output_dir}/{model_name}/training_history.csv")
history = df_history.to_dict(orient='list')


Testing set loss and accuracy

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

In [ ]:
acc = history['accuracy']
val_acc = history['val_accuracy']
loss = history['loss']
val_loss = history['val_loss']
epochs_range = range(len(acc))

training_validation_plots = plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, acc, label='Train Acc')
plt.plot(epochs_range, val_acc, label='Val Acc')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
# salva plot su file
training_validation_plots.savefig(f"{output_dir}/{model_name}/training_validation_plots.png")

In [ ]:
# Confusion Matrix
# Confusion Matrix
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        # inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy()) # Replace with all_labels.extend(labels.numpy()) if using CPU

cm = confusion_matrix(all_labels, all_preds, normalize='true')
confusion_matrix_plot = plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# salva plot su file
confusion_matrix_plot.savefig(f"{output_dir}/{model_name}/confusion_matrix.png")

In [ ]:
# --- 3. Classification Report ---
print("Classification Report:")
report_dict = classification_report(all_labels, all_preds, target_names=le.classes_, labels=np.arange(40), digits=4, zero_division=0)
print(report_dict)

In [ ]:
# Salva classification report su file
with open(f"{output_dir}/{model_name}/classification_report.txt", "w") as f:
    f.write(report_dict)
